# ALM Cash Flow Projections from Cause-Specific Cox Hazards

This notebook converts the cause-specific Cox model outputs (notebook 05) into projected mortgage cash flows for Asset-Liability Management (ALM) analysis.

## Overview
1. **Setup**: Load fitted models and prepare loan data
2. **Baseline hazard validation**: Extract and visualize discrete hazards
3. **Backtest: Predicted vs Realized Cash Flows (Fold 10)**: Out-of-sample validation on held-out test fold
4. **Single loan walkthrough**: Full algorithm on one loan
5. **Portfolio projection**: Aggregate monthly cash flows under base scenario
6. **Scenario analysis**: Rate shocks, HPI stress, recession scenarios
7. **Risk metrics comparison**: NPV, duration, convexity across scenarios
8. **Portfolio segmentation**: Risk metrics by vintage, FICO, LTV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

from src.alm.baseline_hazard import extract_baseline_hazard, extract_baseline_hazards_both
from src.alm.scenarios import (
    MacroScenario, create_base_scenario, apply_rate_shock,
    apply_hpi_shock, apply_unemployment_shock, scenario_to_covariate_matrix,
)
from src.alm.cash_flow_engine import CashFlowConfig, MortgageCashFlowEngine
from src.alm.risk_metrics import (
    compute_npv, compute_modified_duration, compute_modified_convexity,
    compute_wal, compute_all_risk_metrics,
    compute_effective_duration, compute_effective_convexity,
)

sns.set_style('whitegrid')
%matplotlib inline

DATA_DIR = Path('../data/processed')
EXTERNAL_DIR = Path('../data/external')
MODELS_DIR = Path('../models')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports complete.')

---

## 1. Setup: Load Models and Data

In [ ]:
# Load fitted Cox models
with open(MODELS_DIR / 'cox_prepay_tv.pkl', 'rb') as f:
    ctv_prepay = pickle.load(f)
with open(MODELS_DIR / 'cox_default_tv.pkl', 'rb') as f:
    ctv_default = pickle.load(f)

# Feature names from the models
feature_names = ctv_prepay.params_.index.tolist()
print(f'Model features ({len(feature_names)}):')
for i, f in enumerate(feature_names):
    print(f'  {i+1:2d}. {f}: prepay={ctv_prepay.params_[f]:.4f}, default={ctv_default.params_[f]:.4f}')

In [ ]:
# Load panel data
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
print(f'Panel: {len(panel_df):,} loan-months, {panel_df["loan_sequence_number"].nunique():,} loans')

# Load survival data for orig_loan_term and origination-time macro values
surv_df = pd.read_parquet(DATA_DIR / 'survival_data_blumenstock.parquet')

# Get loan-level static data: last observation per loan from panel + join orig_loan_term
last_obs = panel_df.groupby('loan_sequence_number').last().reset_index()

# Join orig_loan_term and origination-time macro from survival data
orig_cols = ['loan_sequence_number', 'orig_loan_term', 'orig_MORTGAGE30US', 'orig_DGS10', 'orig_state_hpi']
orig_info = surv_df[orig_cols].drop_duplicates(subset=['loan_sequence_number'])
last_obs = last_obs.merge(orig_info, on='loan_sequence_number', how='left')

# Fill missing orig_loan_term (default 360 months)
last_obs['orig_loan_term'] = last_obs['orig_loan_term'].fillna(360).astype(int)

# current_loan_age = last observed loan_age (stop value)
last_obs['current_loan_age'] = last_obs['stop'].astype(int)

print(f'\nLoans prepared: {len(last_obs):,}')
print(f'orig_loan_term distribution:')
print(last_obs['orig_loan_term'].value_counts().head())

In [ ]:
# Load macro data
macro_df = pd.read_parquet(EXTERNAL_DIR / 'fred_monthly_panel.parquet')
state_hpi_df = pd.read_parquet(EXTERNAL_DIR / 'state_hpi.parquet')
state_unemp_df = pd.read_parquet(EXTERNAL_DIR / 'state_unemployment.parquet')

print(f'Macro data: {len(macro_df)} months, last: {macro_df.index[-1].strftime("%Y-%m")}')
print(f'State HPI: {state_hpi_df.shape[1]} states, last: {state_hpi_df.index[-1].strftime("%Y-%m")}')
print(f'State unemp: {state_unemp_df.shape[1]} states, last: {state_unemp_df.index[-1].strftime("%Y-%m")}')
print(f'\nLast observed rates:')
print(f'  MORTGAGE30US: {macro_df["MORTGAGE30US"].iloc[-1]:.2f}%')
print(f'  DGS10: {macro_df["DGS10"].iloc[-1]:.2f}%')
print(f'  DGS3MO: {macro_df["DGS3MO"].iloc[-1]:.2f}%')

---

## 2. Baseline Hazard Validation

In [ ]:
# Extract baseline hazards
h0_prepay, h0_default = extract_baseline_hazards_both(ctv_prepay, ctv_default, max_month=360)

print(f'Baseline hazard shapes: prepay={h0_prepay.shape}, default={h0_default.shape}')
print(f'\nPrepay h0: max={h0_prepay.max():.6f}, sum={h0_prepay.sum():.4f}')
print(f'Default h0: max={h0_default.max():.6f}, sum={h0_default.sum():.4f}')

# Cumulative hazard check
H0_prepay = np.cumsum(h0_prepay)
H0_default = np.cumsum(h0_default)

# Baseline-only CIF (no covariates = average risk)
S0 = np.exp(-(H0_prepay + H0_default))
print(f'\nBaseline survival at 120m: {S0[119]:.4f}')
print(f'Baseline survival at 184m: {S0[183]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Discrete hazard h_0(t)
months = np.arange(1, 361)
axes[0, 0].plot(months[:200], h0_prepay[:200], color='steelblue', lw=1.5)
axes[0, 0].set_title('Baseline Discrete Hazard: Prepayment')
axes[0, 0].set_xlabel('Loan Age (months)')
axes[0, 0].set_ylabel('h₀(t)')
axes[0, 0].axvline(184, color='gray', ls='--', alpha=0.5, label='Max observed (184m)')
axes[0, 0].legend()

axes[0, 1].plot(months[:200], h0_default[:200], color='indianred', lw=1.5)
axes[0, 1].set_title('Baseline Discrete Hazard: Default')
axes[0, 1].set_xlabel('Loan Age (months)')
axes[0, 1].set_ylabel('h₀(t)')
axes[0, 1].axvline(170, color='gray', ls='--', alpha=0.5, label='Max observed (170m)')
axes[0, 1].legend()

# Cumulative hazard H_0(t)
axes[1, 0].plot(months[:200], H0_prepay[:200], color='steelblue', lw=1.5)
axes[1, 0].set_title('Baseline Cumulative Hazard: Prepayment')
axes[1, 0].set_xlabel('Loan Age (months)')
axes[1, 0].set_ylabel('H₀(t)')

axes[1, 1].plot(months[:200], H0_default[:200], color='indianred', lw=1.5)
axes[1, 1].set_title('Baseline Cumulative Hazard: Default')
axes[1, 1].set_xlabel('Loan Age (months)')
axes[1, 1].set_ylabel('H₀(t)')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_baseline_hazards.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. Backtest: Predicted vs Realized Cash Flows (Fold 10)

The Cox models were trained on folds 0–9. Fold 10 (9,918 loans) was held out entirely.
Here we use the **actual observed covariates** from the test panel to compute the model's
predicted hazards, survival, and cash flows at each loan-month — then compare against
what actually happened.

**Predicted** (model): At each loan-month, weight the scheduled cash flow by the
model's survival probability S(t-1), and add probability-weighted prepayment and
recovery flows using the sub-density functions f_prepay(t) and f_default(t).

**Realized** (data): The actual cash flow stream — scheduled payments while active,
full UPB payoff at prepayment, recovery at default.

In [ ]:
# === Prepare fold-10 test panel ===
LGD = 0.25

test_panel = panel_df[panel_df['fold'] == 10].copy()

# Join orig_loan_term
test_panel = test_panel.merge(
    surv_df[['loan_sequence_number', 'orig_loan_term']].drop_duplicates(subset=['loan_sequence_number']),
    on='loan_sequence_number', how='left',
)
test_panel['orig_loan_term'] = test_panel['orig_loan_term'].fillna(360).astype(int)

# Drop rows with missing features (< 0.2% of data)
test_panel = test_panel.dropna(subset=feature_names).copy()

n_test_loans = test_panel['loan_sequence_number'].nunique()
print(f'Test panel (fold 10): {len(test_panel):,} loan-months, {n_test_loans:,} loans')
print(f'  Prepayments: {((test_panel["event"]==1) & (test_panel["event_code"]==1)).sum():,}')
print(f'  Defaults: {((test_panel["event"]==1) & (test_panel["event_code"]==2)).sum():,}')

# === Amortization schedule ===
monthly_rate = test_panel['int_rate'].values / 100.0 / 12.0
orig_upb = test_panel['orig_upb'].values
term = test_panel['orig_loan_term'].values
loan_age = test_panel['loan_age'].values

payment = np.where(
    monthly_rate > 0,
    orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
    orig_upb / term,
)

# UPB before and after scheduled payment at this loan_age
factor_prev = (1.0 + monthly_rate) ** (loan_age - 1)
upb_before = np.where(
    monthly_rate > 0,
    orig_upb * factor_prev - payment * (factor_prev - 1.0) / monthly_rate,
    orig_upb - payment * (loan_age - 1),
)
upb_before = np.maximum(upb_before, 0.0)

factor_curr = (1.0 + monthly_rate) ** loan_age
upb_after = np.where(
    monthly_rate > 0,
    orig_upb * factor_curr - payment * (factor_curr - 1.0) / monthly_rate,
    orig_upb - payment * loan_age,
)
upb_after = np.maximum(upb_after, 0.0)

interest_t = upb_before * monthly_rate
principal_t = np.maximum(np.minimum(payment - interest_t, upb_before), 0.0)

print(f'\nAmortization computed for {len(test_panel):,} loan-months')

In [ ]:
# === Compute model-predicted hazards on actual covariates ===
X_test = test_panel[feature_names].values.astype(np.float64)
beta_prepay = ctv_prepay.params_.values.astype(np.float64)
beta_default = ctv_default.params_.values.astype(np.float64)

# Linear predictor and relative risk
lp_prepay = X_test @ beta_prepay
lp_default = X_test @ beta_default
rr_prepay = np.exp(np.clip(lp_prepay, -20, 20))
rr_default = np.exp(np.clip(lp_default, -20, 20))

# Cause-specific hazards: h(t|X) = h0(loan_age) * exp(X*beta)
ages_idx = np.clip(loan_age - 1, 0, len(h0_prepay) - 1)
h_prepay = h0_prepay[ages_idx] * rr_prepay
h_default = h0_default[np.clip(loan_age - 1, 0, len(h0_default) - 1)] * rr_default

# Clip total hazard and rescale proportionally
h_sum = h_prepay + h_default
scale = np.where(h_sum > 0.999, 0.999 / h_sum, 1.0)
h_prepay *= scale
h_default *= scale
h_total = h_prepay + h_default

# Per-loan survival: S(t) = prod_{s=1}^{t} (1 - h_total(s))
test_panel['h_prepay'] = h_prepay
test_panel['h_default'] = h_default
test_panel['log_1mh'] = np.log(1.0 - h_total)
test_panel['cum_log_surv'] = test_panel.groupby('loan_sequence_number')['log_1mh'].cumsum()
test_panel['survival'] = np.exp(test_panel['cum_log_surv'])
test_panel['survival_prev'] = (
    test_panel.groupby('loan_sequence_number')['survival'].shift(1).fillna(1.0)
)

# Sub-densities
test_panel['f_prepay'] = test_panel['h_prepay'] * test_panel['survival_prev']
test_panel['f_default'] = test_panel['h_default'] * test_panel['survival_prev']

# === Predicted cash flows (probability-weighted) ===
test_panel['pred_interest'] = test_panel['survival_prev'].values * interest_t
test_panel['pred_principal'] = test_panel['survival_prev'].values * principal_t
test_panel['pred_prepay'] = test_panel['f_prepay'].values * upb_after
test_panel['pred_recovery'] = test_panel['f_default'].values * upb_after * (1 - LGD)
test_panel['pred_loss'] = test_panel['f_default'].values * upb_after * LGD
test_panel['pred_total'] = (
    test_panel['pred_interest'] + test_panel['pred_principal']
    + test_panel['pred_prepay'] + test_panel['pred_recovery']
)

# === Realized cash flows ===
is_prepay = ((test_panel['event'] == 1) & (test_panel['event_code'] == 1)).values
is_default = ((test_panel['event'] == 1) & (test_panel['event_code'] == 2)).values

test_panel['real_interest'] = np.where(~is_default, interest_t, 0.0)
test_panel['real_principal'] = np.where(~is_default, principal_t, 0.0)
test_panel['real_prepay'] = np.where(is_prepay, upb_after, 0.0)
test_panel['real_recovery'] = np.where(is_default, upb_after * (1 - LGD), 0.0)
test_panel['real_loss'] = np.where(is_default, upb_after * LGD, 0.0)
test_panel['real_total'] = (
    test_panel['real_interest'] + test_panel['real_principal']
    + test_panel['real_prepay'] + test_panel['real_recovery']
)

print('Predicted and realized cash flows computed.')
print(f'\n=== Totals over observation window ===')
for component in ['interest', 'principal', 'prepay', 'recovery', 'loss', 'total']:
    pred = test_panel[f'pred_{component}'].sum()
    real = test_panel[f'real_{component}'].sum()
    ratio = pred / real if real != 0 else float('nan')
    print(f'  {component:>10s}:  predicted ${pred/1e6:>8.2f}M   realized ${real/1e6:>8.2f}M   ratio {ratio:.3f}')

In [ ]:
# === Aggregate by loan_age and compare ===
agg = test_panel.groupby('loan_age').agg(
    n_at_risk=('loan_sequence_number', 'count'),
    pred_total=('pred_total', 'sum'),
    real_total=('real_total', 'sum'),
    pred_interest=('pred_interest', 'sum'),
    real_interest=('real_interest', 'sum'),
    pred_prepay=('pred_prepay', 'sum'),
    real_prepay=('real_prepay', 'sum'),
    pred_loss=('pred_loss', 'sum'),
    real_loss=('real_loss', 'sum'),
    # Event counts and predicted sub-densities for CIF comparison
    f_prepay_sum=('f_prepay', 'sum'),
    f_default_sum=('f_default', 'sum'),
    n_prepay=('event_code', lambda x: (x == 1).sum()),
    n_default=('event_code', lambda x: (x == 2).sum()),
).reset_index()

# Cumulative cash flows
agg['pred_cumul'] = agg['pred_total'].cumsum()
agg['real_cumul'] = agg['real_total'].cumsum()

# Cumulative incidence: predicted (sum of sub-densities) vs realized (cumulative event count / N)
agg['pred_cif_prepay'] = agg['f_prepay_sum'].cumsum() / n_test_loans
agg['pred_cif_default'] = agg['f_default_sum'].cumsum() / n_test_loans
agg['real_cif_prepay'] = agg['n_prepay'].cumsum() / n_test_loans
agg['real_cif_default'] = agg['n_default'].cumsum() / n_test_loans

# Predicted vs realized hazard rate per period
agg['pred_hazard_prepay'] = agg['f_prepay_sum'] / agg['n_at_risk']
agg['real_hazard_prepay'] = agg['n_prepay'] / agg['n_at_risk']
agg['pred_hazard_default'] = agg['f_default_sum'] / agg['n_at_risk']
agg['real_hazard_default'] = agg['n_default'] / agg['n_at_risk']

print(f'Aggregated by loan_age: {len(agg)} time points')
print(f'Max loan_age with >= 100 loans at risk: {agg[agg["n_at_risk"] >= 100]["loan_age"].max()}')

In [ ]:
# === Plots: Predicted vs Realized ===
# Filter to loan_ages with meaningful sample size
mask = agg['n_at_risk'] >= 50
agg_plot = agg[mask]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. CIF: Prepayment
ax = axes[0, 0]
ax.plot(agg_plot['loan_age'], agg_plot['real_cif_prepay'], 'k-', lw=2, label='Realized')
ax.plot(agg_plot['loan_age'], agg_plot['pred_cif_prepay'], '--', color='steelblue', lw=2, label='Predicted')
ax.set_title('Cumulative Incidence: Prepayment')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('CIF')
ax.legend()

# 2. CIF: Default
ax = axes[0, 1]
ax.plot(agg_plot['loan_age'], agg_plot['real_cif_default'], 'k-', lw=2, label='Realized')
ax.plot(agg_plot['loan_age'], agg_plot['pred_cif_default'], '--', color='indianred', lw=2, label='Predicted')
ax.set_title('Cumulative Incidence: Default')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('CIF')
ax.legend()

# 3. Monthly total cash flow
ax = axes[0, 2]
ax.plot(agg_plot['loan_age'], agg_plot['real_total'] / 1e6, 'k-', lw=1.5, alpha=0.7, label='Realized')
ax.plot(agg_plot['loan_age'], agg_plot['pred_total'] / 1e6, '--', color='steelblue', lw=1.5, label='Predicted')
ax.set_title('Monthly Total Cash Flow')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('$M')
ax.legend()

# 4. Cumulative cash flow
ax = axes[1, 0]
ax.plot(agg_plot['loan_age'], agg_plot['real_cumul'] / 1e6, 'k-', lw=2, label='Realized')
ax.plot(agg_plot['loan_age'], agg_plot['pred_cumul'] / 1e6, '--', color='steelblue', lw=2, label='Predicted')
ax.set_title('Cumulative Cash Flow')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('$M (cumulative)')
ax.legend()

# 5. Prepayment hazard rate (smoothed)
ax = axes[1, 1]
window = 6
ax.plot(agg_plot['loan_age'],
        agg_plot['real_hazard_prepay'].rolling(window, center=True, min_periods=1).mean(),
        'k-', lw=1.5, label=f'Realized ({window}m avg)')
ax.plot(agg_plot['loan_age'],
        agg_plot['pred_hazard_prepay'].rolling(window, center=True, min_periods=1).mean(),
        '--', color='steelblue', lw=1.5, label=f'Predicted ({window}m avg)')
ax.set_title('Prepayment Hazard Rate')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('h(t)')
ax.legend()

# 6. Residual (predicted - realized) cumulative CF
ax = axes[1, 2]
residual = (agg_plot['pred_cumul'] - agg_plot['real_cumul']) / 1e6
ax.plot(agg_plot['loan_age'], residual, '-', color='darkred', lw=1.5)
ax.axhline(0, color='black', ls='--', lw=0.8)
ax.set_title('Cumulative CF Residual (Pred - Real)')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('$M')
ax.fill_between(agg_plot['loan_age'], 0, residual, alpha=0.2, color='darkred')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Backtest: Predicted vs Realized (Fold 10, Out-of-Sample)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_backtest_predicted_vs_realized.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Summary statistics ===
# Compare at key horizons
print('=== Backtest Summary: Predicted vs Realized (Fold 10) ===\n')

horizons = [24, 48, 72, 120, 150]
print(f'{"Horizon":>8s}  {"Pred CIF_P":>10s}  {"Real CIF_P":>10s}  {"Pred CIF_D":>10s}  {"Real CIF_D":>10s}  {"Pred CF ($M)":>13s}  {"Real CF ($M)":>13s}')
print('-' * 90)
for h in horizons:
    row = agg[agg['loan_age'] == h]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    print(f'{h:>6d}m  {row["pred_cif_prepay"]:>10.4f}  {row["real_cif_prepay"]:>10.4f}  '
          f'{row["pred_cif_default"]:>10.4f}  {row["real_cif_default"]:>10.4f}  '
          f'{row["pred_cumul"]/1e6:>13.2f}  {row["real_cumul"]/1e6:>13.2f}')

# Overall prediction error
total_pred = test_panel['pred_total'].sum()
total_real = test_panel['real_total'].sum()
mape_monthly = np.mean(np.abs(agg_plot['pred_total'] - agg_plot['real_total']) / agg_plot['real_total'].clip(lower=1)) * 100

print(f'\n=== Aggregate Accuracy ===')
print(f'Total predicted CF:  ${total_pred/1e6:,.2f}M')
print(f'Total realized CF:   ${total_real/1e6:,.2f}M')
print(f'Prediction error:    ${(total_pred - total_real)/1e6:,.2f}M ({(total_pred/total_real - 1)*100:+.2f}%)')
print(f'Monthly MAPE:        {mape_monthly:.1f}%')

# === Present value of cash flow difference ===
# Discount rate: use the last observed mortgage rate
disc_rate_annual = macro_df['MORTGAGE30US'].iloc[-1] / 100.0
disc_rate_monthly = disc_rate_annual / 12.0

# Compute PV of predicted and realized CF streams (aggregated by loan_age)
t_months = agg['loan_age'].values
discount_factors = (1.0 + disc_rate_monthly) ** (-t_months)

pv_pred = np.sum(agg['pred_total'].values * discount_factors)
pv_real = np.sum(agg['real_total'].values * discount_factors)
pv_diff = pv_pred - pv_real

# Portfolio size: total orig UPB of fold-10 loans
test_orig_upb = test_panel.groupby('loan_sequence_number')['orig_upb'].first().sum()

print(f'\n=== Present Value of Cash Flow Error ===')
print(f'Discount rate:       {disc_rate_annual*100:.2f}% (annual)')
print(f'PV predicted CF:     ${pv_pred/1e6:,.2f}M')
print(f'PV realized CF:      ${pv_real/1e6:,.2f}M')
print(f'PV difference:       ${pv_diff/1e6:,.2f}M ({pv_diff/pv_real*100:+.2f}% of PV realized)')
print(f'Portfolio orig UPB:  ${test_orig_upb/1e6:,.2f}M')
print(f'PV error / UPB:      {pv_diff/test_orig_upb*100:+.3f}%')

# CIF error at last common observation
last_age = agg_plot['loan_age'].max()
last_row = agg[agg['loan_age'] == last_age].iloc[0]
print(f'\nAt loan_age {last_age}:')
print(f'  Prepay CIF error:  {last_row["pred_cif_prepay"] - last_row["real_cif_prepay"]:+.4f}')
print(f'  Default CIF error: {last_row["pred_cif_default"] - last_row["real_cif_default"]:+.4f}')

---

## 4. Single Loan Walkthrough

In [ ]:
# Pick a representative loan
# Choose a 30-year loan that is still active (censored)
active_loans = last_obs[
    (last_obs['orig_loan_term'] == 360) &
    (last_obs['event_code'] == 0) &
    (last_obs['current_loan_age'] > 12) &
    (last_obs['orig_MORTGAGE30US'].notna())
]
example_loan = active_loans.iloc[0:1].copy()

print('=== Example Loan ===')
for col in ['loan_sequence_number', 'int_rate', 'orig_upb', 'fico_score', 'dti_r', 'ltv_r',
            'orig_loan_term', 'current_loan_age', 'property_state',
            'orig_MORTGAGE30US', 'orig_DGS10', 'orig_state_hpi']:
    if col in example_loan.columns:
        print(f'  {col}: {example_loan[col].iloc[0]}')

In [ ]:
# Create base scenario
base_scenario = create_base_scenario(
    panel_df=panel_df,
    macro_df=macro_df,
    state_hpi_df=state_hpi_df,
    state_unemp_df=state_unemp_df,
    horizon=360,
)
print(f'Base scenario: {base_scenario.name}')
print(f'  Horizon: {base_scenario.horizon} months')
print(f'  MORTGAGE30US: {base_scenario.mortgage30us[0]:.2f}% (flat)')
print(f'  DGS10: {base_scenario.dgs10[0]:.2f}% (flat)')
print(f'  States with HPI: {len(base_scenario.state_hpi)}')
print(f'  States with unemp: {len(base_scenario.state_unemployment)}')

In [ ]:
# Build covariate matrix for single loan
X_single = scenario_to_covariate_matrix(
    loans_df=example_loan,
    scenario=base_scenario,
    macro_df=macro_df,
    state_hpi_df=state_hpi_df,
    state_unemp_df=state_unemp_df,
    feature_names=feature_names,
)
print(f'Covariate matrix shape: {X_single.shape}  (loans x months x features)')

# Show first few months of key features
print('\nFirst 6 months of key features:')
key_feats = ['int_rate', 'orig_upb', 'bal_repaid', 'ppi_c_FRMA', 't_act_12m']
# Fall back gracefully if model uses log_upb instead of orig_upb
key_feats = [f for f in key_feats if f in feature_names]
for feat in key_feats:
    idx = feature_names.index(feat)
    vals = X_single[0, :6, idx]
    print(f'  {feat}: {[f"{v:.3f}" for v in vals]}')

In [ ]:
# Project cash flows for single loan
config = CashFlowConfig(lgd=0.25)
engine = MortgageCashFlowEngine(
    h0_prepay=h0_prepay,
    h0_default=h0_default,
    beta_prepay=ctv_prepay.params_.values,
    beta_default=ctv_default.params_.values,
    config=config,
)

cf_single = engine.project_cash_flows(example_loan, X_single)

# Verify accounting identity: S(T) + CIF_prepay(T) + CIF_default(T) ≈ 1
T = cf_single['survival'].shape[1]
cif_prepay = np.cumsum(cf_single['f_prepay'][0])
cif_default = np.cumsum(cf_single['f_default'][0])
identity_check = cf_single['survival'][0, -1] + cif_prepay[-1] + cif_default[-1]

print(f'=== Single Loan Cash Flow Summary ===')
print(f'Projection horizon: {T} months')
print(f'Total interest: ${cf_single["interest"][0].sum():,.2f}')
print(f'Total sched principal: ${cf_single["scheduled_principal"][0].sum():,.2f}')
print(f'Total prepayment: ${cf_single["prepayment"][0].sum():,.2f}')
print(f'Total recovery: ${cf_single["recovery"][0].sum():,.2f}')
print(f'Total loss: ${cf_single["loss"][0].sum():,.2f}')
print(f'Total CF: ${cf_single["total_cf"][0].sum():,.2f}')
print(f'\n=== Accounting Identity Check ===')
print(f'S({T}): {cf_single["survival"][0, -1]:.6f}')
print(f'CIF_prepay({T}): {cif_prepay[-1]:.6f}')
print(f'CIF_default({T}): {cif_default[-1]:.6f}')
print(f'Sum: {identity_check:.6f} (should be ~1.0)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
months = np.arange(1, T + 1)

# Survival and CIF
ax = axes[0, 0]
ax.plot(months, cf_single['survival'][0], 'k-', lw=2, label='Survival S(t)')
ax.plot(months, cif_prepay, '--', color='steelblue', lw=1.5, label='CIF prepay')
ax.plot(months, cif_default, '--', color='indianred', lw=1.5, label='CIF default')
ax.fill_between(months, 0, cif_default, alpha=0.2, color='indianred')
ax.fill_between(months, cif_default, cif_default + cif_prepay, alpha=0.2, color='steelblue')
ax.set_title('Survival and Cumulative Incidence')
ax.set_xlabel('Month')
ax.set_ylabel('Probability')
ax.legend()

# Expected cash flows
ax = axes[0, 1]
ax.stackplot(months,
    cf_single['interest'][0],
    cf_single['scheduled_principal'][0],
    cf_single['prepayment'][0],
    cf_single['recovery'][0],
    labels=['Interest', 'Sched Principal', 'Prepayment', 'Recovery'],
    colors=['#4e79a7', '#59a14f', '#f28e2b', '#e15759'],
    alpha=0.8)
ax.set_title('Expected Monthly Cash Flows')
ax.set_xlabel('Month')
ax.set_ylabel('$')
ax.legend(loc='upper right', fontsize=8)

# Hazard rates
ax = axes[1, 0]
# Compute realized hazards from sub-densities and survival
s_start = np.ones(T)
s_start[1:] = cf_single['survival'][0, :-1]
h_prepay_realized = np.where(s_start > 1e-10, cf_single['f_prepay'][0] / s_start, 0)
h_default_realized = np.where(s_start > 1e-10, cf_single['f_default'][0] / s_start, 0)
ax.plot(months, h_prepay_realized, color='steelblue', lw=1.5, label='h_prepay(t)')
ax.plot(months, h_default_realized, color='indianred', lw=1.5, label='h_default(t)')
ax.set_title('Cause-Specific Hazard Rates')
ax.set_xlabel('Month')
ax.set_ylabel('Hazard h(t)')
ax.legend()

# Amortization
ax = axes[1, 1]
ax.plot(months, cf_single['upb_schedule'][0], 'k-', lw=1.5, label='Scheduled UPB')
ax.set_title('Amortization Schedule')
ax.set_xlabel('Month')
ax.set_ylabel('UPB ($)')
ax.legend()

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Single Loan Walkthrough: {example_loan["loan_sequence_number"].iloc[0]}', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_single_loan_walkthrough.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 5. Portfolio Projection (Base Scenario)

In [ ]:
# Prepare portfolio: filter to loans with complete origination data
portfolio = last_obs[
    last_obs['orig_MORTGAGE30US'].notna() &
    last_obs['orig_DGS10'].notna() &
    last_obs['orig_state_hpi'].notna()
].copy()

# Limit remaining term: only project up to remaining term
portfolio['remaining_term'] = portfolio['orig_loan_term'] - portfolio['current_loan_age']
portfolio = portfolio[portfolio['remaining_term'] > 0].copy()

print(f'Portfolio: {len(portfolio):,} loans')
print(f'Total orig UPB: ${portfolio["orig_upb"].sum():,.0f}')
print(f'Average remaining term: {portfolio["remaining_term"].mean():.0f} months')
print(f'\nBy orig_loan_term:')
print(portfolio.groupby('orig_loan_term').agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_rate=('int_rate', 'mean'),
    avg_fico=('fico_score', 'mean'),
    avg_age=('current_loan_age', 'mean'),
).round(1))

In [ ]:
%%time
# Build covariate matrix for full portfolio under base scenario
print('Building covariate matrix...')
X_base = scenario_to_covariate_matrix(
    loans_df=portfolio,
    scenario=base_scenario,
    macro_df=macro_df,
    state_hpi_df=state_hpi_df,
    state_unemp_df=state_unemp_df,
    feature_names=feature_names,
)
print(f'Covariate matrix: {X_base.shape} ({X_base.nbytes / 1e6:.1f} MB)')

# Project cash flows
print('Projecting cash flows...')
cf_base = engine.project_cash_flows(portfolio, X_base)

# Aggregate to portfolio level
portfolio_cf = engine.aggregate_portfolio(cf_base)

print(f'\n=== Portfolio Cash Flow Summary (Base Scenario) ===')
print(f'Total expected CF: ${portfolio_cf["total_cf"].sum():,.0f}')
print(f'Total interest: ${portfolio_cf["interest"].sum():,.0f}')
print(f'Total sched principal: ${portfolio_cf["scheduled_principal"].sum():,.0f}')
print(f'Total prepayment: ${portfolio_cf["prepayment"].sum():,.0f}')
print(f'Total recovery: ${portfolio_cf["recovery"].sum():,.0f}')
print(f'Total loss: ${portfolio_cf["loss"].sum():,.0f}')
print(f'\nAvg survival at end: {portfolio_cf["avg_survival"].iloc[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

months = portfolio_cf['month'].values

# Stacked cash flows
ax = axes[0, 0]
ax.stackplot(months,
    portfolio_cf['interest'] / 1e6,
    portfolio_cf['scheduled_principal'] / 1e6,
    portfolio_cf['prepayment'] / 1e6,
    portfolio_cf['recovery'] / 1e6,
    labels=['Interest', 'Sched Principal', 'Prepayment', 'Recovery'],
    colors=['#4e79a7', '#59a14f', '#f28e2b', '#e15759'],
    alpha=0.8)
ax.set_title('Portfolio Monthly Cash Flows')
ax.set_xlabel('Month')
ax.set_ylabel('$M')
ax.legend(loc='upper right', fontsize=8)

# Cumulative cash flows
ax = axes[0, 1]
ax.plot(months, portfolio_cf['cumulative_cf'] / 1e6, 'k-', lw=2)
ax.set_title('Cumulative Portfolio Cash Flows')
ax.set_xlabel('Month')
ax.set_ylabel('$M (cumulative)')

# Average survival
ax = axes[1, 0]
ax.plot(months, portfolio_cf['avg_survival'], 'k-', lw=2)
ax.set_title('Average Portfolio Survival Rate')
ax.set_xlabel('Month')
ax.set_ylabel('S(t)')

# Loss over time
ax = axes[1, 1]
ax.plot(months, portfolio_cf['loss'].cumsum() / 1e6, 'r-', lw=2)
ax.set_title('Cumulative Expected Loss')
ax.set_xlabel('Month')
ax.set_ylabel('$M (cumulative)')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Portfolio Cash Flow Projection - Base Scenario', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_portfolio_base_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute risk metrics for base scenario
# Use mortgage rate as discount rate
discount_rate = macro_df['MORTGAGE30US'].iloc[-1] / 100.0

base_metrics = compute_all_risk_metrics(portfolio_cf, annual_rate=discount_rate)

print(f'=== Base Scenario Risk Metrics ===')
print(f'Discount rate: {discount_rate*100:.2f}%')
print(f'NPV: ${base_metrics["npv"]:,.0f}')
print(f'Modified Duration: {base_metrics["modified_duration"]:.2f} years')
print(f'Modified Convexity: {base_metrics["modified_convexity"]:.2f}')
print(f'WAL: {base_metrics["wal_years"]:.2f} years')
print(f'Total Cash Flow: ${base_metrics["total_cash_flow"]:,.0f}')
print(f'Total Loss: ${base_metrics["total_loss"]:,.0f}')

---

## 6. Scenario Analysis

In [ ]:
# Define scenarios
scenarios = {
    'base': base_scenario,
    'rate_+100bp': apply_rate_shock(base_scenario, +100),
    'rate_+200bp': apply_rate_shock(base_scenario, +200),
    'rate_-100bp': apply_rate_shock(base_scenario, -100),
    'rate_-200bp': apply_rate_shock(base_scenario, -200),
    'hpi_stress': apply_hpi_shock(base_scenario, decline_pct=20, decline_months=24, name='hpi_stress'),
    'recession': MacroScenario(
        name='recession',
        mortgage30us=apply_rate_shock(base_scenario, -150).mortgage30us,
        dgs10=apply_rate_shock(base_scenario, -150).dgs10,
        dgs3mo=apply_rate_shock(base_scenario, -200).dgs3mo,
        state_hpi=apply_hpi_shock(base_scenario, decline_pct=15, decline_months=18).state_hpi,
        state_unemployment=apply_unemployment_shock(base_scenario, increase_pct_pts=3, ramp_months=12).state_unemployment,
        national_hpi=apply_hpi_shock(base_scenario, decline_pct=15, decline_months=18).national_hpi,
    ),
    'severe_recession': MacroScenario(
        name='severe_recession',
        mortgage30us=apply_rate_shock(base_scenario, -200).mortgage30us,
        dgs10=apply_rate_shock(base_scenario, -200).dgs10,
        dgs3mo=apply_rate_shock(base_scenario, -300).dgs3mo,
        state_hpi=apply_hpi_shock(base_scenario, decline_pct=30, decline_months=36).state_hpi,
        state_unemployment=apply_unemployment_shock(base_scenario, increase_pct_pts=6, ramp_months=18).state_unemployment,
        national_hpi=apply_hpi_shock(base_scenario, decline_pct=30, decline_months=36).national_hpi,
    ),
}

print(f'Defined {len(scenarios)} scenarios:')
for name, scen in scenarios.items():
    print(f'  {name}: MORTGAGE30US={scen.mortgage30us[0]:.2f}%, DGS10={scen.dgs10[0]:.2f}%')

In [ ]:
%%time
# Run all scenarios
scenario_results = {}

for name, scen in scenarios.items():
    print(f'Running scenario: {name}...')
    
    # Build covariates
    X_scen = scenario_to_covariate_matrix(
        loans_df=portfolio,
        scenario=scen,
        macro_df=macro_df,
        state_hpi_df=state_hpi_df,
        state_unemp_df=state_unemp_df,
        feature_names=feature_names,
    )
    
    # Project cash flows
    cf_scen = engine.project_cash_flows(portfolio, X_scen)
    agg_scen = engine.aggregate_portfolio(cf_scen)
    
    # Risk metrics
    # For rate-shocked scenarios, also shock the discount rate
    if 'rate_+' in name:
        shock_bps = int(name.split('+')[1].replace('bp', ''))
        disc_rate = discount_rate + shock_bps / 10000
    elif 'rate_-' in name:
        shock_bps = int(name.split('-')[1].replace('bp', ''))
        disc_rate = discount_rate - shock_bps / 10000
    elif name in ('recession', 'severe_recession'):
        # Use the scenario's mortgage rate as discount rate proxy
        disc_rate = scen.mortgage30us[0] / 100.0
    else:
        disc_rate = discount_rate
    
    metrics = compute_all_risk_metrics(agg_scen, annual_rate=disc_rate)
    
    scenario_results[name] = {
        'portfolio_cf': agg_scen,
        'metrics': metrics,
        'discount_rate': disc_rate,
    }
    
    print(f'  NPV: ${metrics["npv"]:,.0f}, Duration: {metrics["modified_duration"]:.2f}y, WAL: {metrics["wal_years"]:.2f}y')

print('\nAll scenarios complete.')

---

## 7. Risk Metrics Comparison

In [ ]:
# Create comparison table
metrics_rows = []
for name, res in scenario_results.items():
    m = res['metrics']
    metrics_rows.append({
        'Scenario': name,
        'Discount Rate': f"{res['discount_rate']*100:.2f}%",
        'NPV ($M)': m['npv'] / 1e6,
        'Mod Duration (yr)': m['modified_duration'],
        'Mod Convexity': m['modified_convexity'],
        'WAL (yr)': m['wal_years'],
        'Total CF ($M)': m['total_cash_flow'] / 1e6,
        'Total Loss ($M)': m['total_loss'] / 1e6,
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Scenario')
print('=== Risk Metrics Comparison ===')
print(metrics_df.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenario_names = list(scenario_results.keys())
x = np.arange(len(scenario_names))

# NPV
ax = axes[0, 0]
npvs = [scenario_results[s]['metrics']['npv'] / 1e6 for s in scenario_names]
colors = ['#4e79a7' if s == 'base' else '#59a14f' if 'rate_-' in s else '#e15759' if 'rate_+' in s else '#f28e2b' for s in scenario_names]
ax.bar(x, npvs, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('NPV by Scenario')
ax.set_ylabel('$M')

# Duration
ax = axes[0, 1]
durations = [scenario_results[s]['metrics']['modified_duration'] for s in scenario_names]
ax.bar(x, durations, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Modified Duration by Scenario')
ax.set_ylabel('Years')

# WAL
ax = axes[1, 0]
wals = [scenario_results[s]['metrics']['wal_years'] for s in scenario_names]
ax.bar(x, wals, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Weighted Average Life by Scenario')
ax.set_ylabel('Years')

# Total Loss
ax = axes[1, 1]
losses = [scenario_results[s]['metrics']['total_loss'] / 1e6 for s in scenario_names]
ax.bar(x, losses, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Total Expected Loss by Scenario')
ax.set_ylabel('$M')

for ax in axes.flat:
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Risk Metrics Across Scenarios', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_scenario_risk_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Overlay portfolio cash flows across key scenarios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

key_scenarios = ['base', 'rate_+200bp', 'rate_-200bp', 'recession', 'severe_recession']
scenario_colors = {'base': 'black', 'rate_+200bp': '#e15759', 'rate_-200bp': '#59a14f',
                   'recession': '#f28e2b', 'severe_recession': '#b07aa1'}

for name in key_scenarios:
    if name in scenario_results:
        cf = scenario_results[name]['portfolio_cf']
        lw = 2.5 if name == 'base' else 1.5
        ls = '-' if name == 'base' else '--'
        
        axes[0].plot(cf['month'], cf['total_cf'] / 1e6,
                     color=scenario_colors.get(name, 'gray'), lw=lw, ls=ls, label=name)
        axes[1].plot(cf['month'], cf['avg_survival'],
                     color=scenario_colors.get(name, 'gray'), lw=lw, ls=ls, label=name)

axes[0].set_title('Monthly Total Cash Flow')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('$M')
axes[0].legend(fontsize=8)

axes[1].set_title('Average Survival')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('S(t)')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_scenario_cf_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Portfolio Segmentation

In [ ]:
# Compute loan-level NPV and WAL under base scenario for segmentation
loan_npv = compute_npv(cf_base['total_cf'], annual_rate=discount_rate)
loan_principal_return = cf_base['scheduled_principal'] + cf_base['prepayment'] + cf_base['recovery']

# Per-loan WAL
T_base = cf_base['total_cf'].shape[1]
t_arr = np.arange(1, T_base + 1)
loan_total_prin = loan_principal_return.sum(axis=1)
loan_wal = np.where(
    loan_total_prin > 1e-6,
    np.sum(t_arr[None, :] * loan_principal_return, axis=1) / loan_total_prin / 12.0,
    0.0
)

# Add to portfolio df
portfolio = portfolio.copy()
portfolio['npv'] = loan_npv
portfolio['wal_years'] = loan_wal

# Create bins
portfolio['fico_band'] = pd.cut(portfolio['fico_score'], bins=[0, 680, 720, 760, 800, 900],
                                 labels=['<680', '680-720', '720-760', '760-800', '800+'])
portfolio['ltv_band'] = pd.cut(portfolio['ltv_r'], bins=[0, 60, 70, 80, 90, 100],
                                labels=['<60', '60-70', '70-80', '80-90', '90+'])

print(f'Loan-level metrics computed for {len(portfolio):,} loans')

In [ ]:
# By vintage
vintage_metrics = portfolio.groupby('vintage_year').agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_rate=('int_rate', 'mean'),
    avg_fico=('fico_score', 'mean'),
    total_upb=('orig_upb', 'sum'),
    total_npv=('npv', 'sum'),
    avg_wal=('wal_years', 'mean'),
).round(2)
vintage_metrics['total_upb_M'] = (vintage_metrics['total_upb'] / 1e6).round(1)
vintage_metrics['total_npv_M'] = (vintage_metrics['total_npv'] / 1e6).round(1)

print('=== Risk Metrics by Vintage ===')
print(vintage_metrics[['n_loans', 'avg_rate', 'avg_fico', 'total_upb_M', 'total_npv_M', 'avg_wal']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By FICO
fico_metrics = portfolio.groupby('fico_band', observed=True).agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_npv=('npv', 'mean'),
    avg_wal=('wal_years', 'mean'),
).reset_index()

ax = axes[0]
x = np.arange(len(fico_metrics))
width = 0.35
ax2 = ax.twinx()
bars = ax.bar(x - width/2, fico_metrics['avg_npv'] / 1e3, width, color='steelblue', alpha=0.8, label='Avg NPV ($K)')
line = ax2.plot(x, fico_metrics['avg_wal'], 'o-', color='indianred', lw=2, label='Avg WAL (yr)')
ax.set_xticks(x)
ax.set_xticklabels(fico_metrics['fico_band'])
ax.set_xlabel('FICO Band')
ax.set_ylabel('Avg NPV ($K)', color='steelblue')
ax2.set_ylabel('Avg WAL (yr)', color='indianred')
ax.set_title('Risk Metrics by FICO')
ax.grid(True, alpha=0.3, axis='y')

# By LTV
ltv_metrics = portfolio.groupby('ltv_band', observed=True).agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_npv=('npv', 'mean'),
    avg_wal=('wal_years', 'mean'),
).reset_index()

ax = axes[1]
x = np.arange(len(ltv_metrics))
ax2 = ax.twinx()
bars = ax.bar(x - width/2, ltv_metrics['avg_npv'] / 1e3, width, color='steelblue', alpha=0.8, label='Avg NPV ($K)')
line = ax2.plot(x, ltv_metrics['avg_wal'], 'o-', color='indianred', lw=2, label='Avg WAL (yr)')
ax.set_xticks(x)
ax.set_xticklabels(ltv_metrics['ltv_band'])
ax.set_xlabel('LTV Band')
ax.set_ylabel('Avg NPV ($K)', color='steelblue')
ax2.set_ylabel('Avg WAL (yr)', color='indianred')
ax.set_title('Risk Metrics by LTV')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute effective duration and convexity (with macro covariate shocks)
# This is the key metric that captures prepayment optionality

def scenario_func(shock_bps):
    """Helper: return (X_shocked, scenario) for a given rate shock."""
    if shock_bps == 0:
        scen = base_scenario
    else:
        scen = apply_rate_shock(base_scenario, shock_bps)
    X = scenario_to_covariate_matrix(
        loans_df=portfolio,
        scenario=scen,
        macro_df=macro_df,
        state_hpi_df=state_hpi_df,
        state_unemp_df=state_unemp_df,
        feature_names=feature_names,
    )
    return X, scen

print('Computing effective duration (100bp shock)...')
eff_dur = compute_effective_duration(
    engine, portfolio, scenario_func, annual_rate=discount_rate, shock_bps=100
)

print('Computing effective convexity (100bp shock)...')
eff_conv = compute_effective_convexity(
    engine, portfolio, scenario_func, annual_rate=discount_rate, shock_bps=100
)

print(f'\n=== Duration/Convexity Comparison ===')
print(f'Modified Duration:  {base_metrics["modified_duration"]:.2f} years')
print(f'Effective Duration: {eff_dur:.2f} years')
print(f'Modified Convexity:  {base_metrics["modified_convexity"]:.2f}')
print(f'Effective Convexity: {eff_conv:.2f}')
print(f'\nEffective < Modified duration indicates negative convexity from prepayment optionality')

---

## Summary

### Key Results
- Converted cause-specific Cox hazard models into projected mortgage cash flows
- **Backtested on held-out fold 10** (9,918 loans): compared predicted vs realized CIF and cash flows
- Validated baseline hazard extraction and single-loan accounting identity
- Projected portfolio-level cash flows under base and 7 stress scenarios
- Computed interest rate risk metrics (NPV, duration, convexity, WAL)
- Analyzed risk metrics by vintage, FICO, and LTV segments

### Sanity Checks
- S(T) + CIF_prepay(T) + CIF_default(T) = 1 (accounting identity)
- Predicted CIF tracks realized CIF on out-of-sample fold 10
- Prepayments accelerate when rates fall, slow when rates rise
- Effective duration < Modified duration (prepayment optionality)
- Default losses increase under HPI stress and recession scenarios